**2.2 --- 
step1.** Load dataset from Hugging Face

In [ ]:
# 1. Install the library we need to download data from Hugging Face
!pip install datasets

# 2. Import the library
from datasets import load_dataset

# 3. Download the specific dataset mentioned in your project
dataset = load_dataset("ReySajju742/Urdu-Poetry-Dataset")

print("Dataset download successful!")
print(dataset)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 37.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.


README.md: 0.00B [00:00, ?B/s]

output_ur.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1323 [00:00<?, ? examples/s]

Dataset download successful!
DatasetDict({
    train: Dataset({
        features: ['title', 'content'],
        num_rows: 1323
    })
})


In [3]:
from collections import Counter
import pandas as pd

# Ensure we are using the dataframe we loaded earlier
df = dataset['train'].to_pandas()

# --- QUESTION 1: How many poems? ---
total_poems = len(df)

# --- QUESTION 2: Missing or Corrupted Entries? ---
# Check if any row in 'content' is empty (Null/None)
missing_count = df['content'].isnull().sum()
# Check for empty strings (poems with length 0)
empty_strings = df[df['content'] == ''].shape[0]

# --- QUESTION 3: Average Length of Poems? ---
# We calculate this in two ways: Lines per poem, and Words per poem
# 1. Count lines (split by new line character)
df['line_count'] = df['content'].apply(lambda x: len(x.split('\n')) if x else 0)
# 2. Count words (split by spaces)
df['word_count'] = df['content'].apply(lambda x: len(x.split()) if x else 0)

avg_lines = df['line_count'].mean()
avg_words = df['word_count'].mean()

# --- QUESTION 4: Most Common Words? ---
# Join all poems into one massive text blob
all_text = " ".join(df['content'].dropna())
# Split into words (basic split for exploration)
all_words = all_text.split()
# Count frequency
word_counts = Counter(all_words)
most_common = word_counts.most_common(10)

# --- PRINT THE REPORT ANSWERS ---
print(f"1. Total Poems: {total_poems}")
print(f"2. Missing/Corrupted Entries: {missing_count} nulls, {empty_strings} empty strings.")
print(f"3. Average Poem Length:")
print(f"   - Average Lines per poem: {avg_lines:.2f}")
print(f"   - Average Words per poem: {avg_words:.2f}")
print(f"4. Top 10 Most Common Words (Raw Data):")
for word, count in most_common:
    print(f"   - {word}: {count}")

1. Total Poems: 1323
2. Missing/Corrupted Entries: 9 nulls, 0 empty strings.
3. Average Poem Length:
   - Average Lines per poem: 16.92
   - Average Words per poem: 130.93
4. Top 10 Most Common Words (Raw Data):
   - ہے: 6831
   - میں: 4674
   - سے: 3669
   - کے: 2863
   - کی: 2774
   - کو: 2680
   - تو: 2669
   - نہ: 2313
   - ہیں: 2263
   - بھی: 2222



**2.2 --- step 2. Extract individual lines from poems**


In [ ]:
import pandas as pd

# Convert the dataset to a Pandas DataFrame (easier to read)
df = dataset['train'].to_pandas()

# Show the first 5 rows
print("First 5 entries:")
display(df.head())

# print ONE full poem to see the structure (newlines, special characters)
print("\n--- SAMPLE POEM CONTENT ---")
print(df['content'][0])

First 5 entries:


,title,content
0,be-thikaane-hai-dil-e-gam-ghiin-thikaane-kii-k...,\nبے ٹھکانے ہے دل غمگیں ٹھکانے کی کہو \nشام ہج...
1,bahsen-chhidii-huii-hain-hayaat-o-mamaat-kii-f...,\nبحثیں چھڑی ہوئی ہیں حیات و ممات کی \nسو بات ...
2,aaj-bhii-qaafila-e-ishq-ravaan-hai-ki-jo-thaa-...,\nآج بھی قافلۂ عشق رواں ہے کہ جو تھا \nوہی میل...
3,tumhen-kyuunkar-bataaen-zindagii-ko-kyaa-samaj...,\nتمہیں کیوں کر بتائیں زندگی کو کیا سمجھتے ہیں...
4,tez-ehsaas-e-khudii-darkaar-hai-firaq-gorakhpu...,\nتیز احساس خودی درکار ہے \nزندگی کو زندگی درک...



--- SAMPLE POEM CONTENT ---

بے ٹھکانے ہے دل غمگیں ٹھکانے کی کہو 
شام ہجراں دوستو کچھ اس کے آنے کی کہو 
ہاں نہ پوچھ اک گرفتار قفس کی زندگی 
ہم صفیران چمن کچھ آشیانے کی کہو 
اڑ گیا ہے منزل دشوار میں غم کا سمند 
گیسوئے پر پیچ و خم کے تازیانے کی کہو 
بات بنتی اور باتوں سے نظر آتی نہیں 
اس نگاہ ناز کی باتیں بنانے کی کہو 
داستاں وہ تھی جسے دل بجھتے بجھتے کہہ گیا 
شمع بزم زندگی کے جھلملانے کی کہو 
کچھ دل مرحوم کی باتیں کرو اے اہل علم 
جس سے ویرانے تھے آباد اس دوانے کی کہو 
داستان زندگی بھی کس قدر دلچسپ ہے 
جو ازل سے چھڑ گیا ہے اس فسانے کی کہو 
یہ فسون نیم شب یہ خواب‌‌ ساماں خامشی 
سامری فن آنکھ کے جادو جگانے کی کہو 
کوئی کیا کھائے گا یوں سچی قسم جھوٹی قسم 
اس نگاہ ناز کی سوگندھ کھانے کی کہو 
شام ہی سے گوش بر آواز ہے بزم سخن 
کچھ فراقؔ اپنی سناؤ کچھ زمانے کی کہو 


**2.2 --- Step 3 and 4 tokenization and vocabulary**

In [ ]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer
import pickle

# --- RE-RUNNING CLEANING (To ensure all_verses is ready) ---
def clean_verse(text):
    text = re.sub(r'[a-zA-Z0-9]', '', text) # Remove English/Numbers
    text = re.sub(r'[()\[\]{}|\\/<>@#$%^&*_+=]', '', text) # Remove symbols
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

# Extract verses
all_verses = []
for poem in df['content']: 
    if poem:
        lines = poem.split('\n')
        for line in lines:
            cleaned = clean_verse(line)
            if len(cleaned) > 5: # Filter tiny lines
                all_verses.append(cleaned)

print(f"Step 2 Complete: Extracted {len(all_verses)} clean verses.")

# --- STEP 3: TOKENIZATION ---
# 1. Initialize Tokenizer
# filters='' becausee already cleaned the text manually
tokenizer = Tokenizer(filters='', lower=False) 

# 2. Build the Vocabulary
tokenizer.fit_on_texts(all_verses)

# 3. Calculate Vocabulary Size
# We add +1 because ID '0' is reserved for padding (empty space)
vocab_size = len(tokenizer.word_index) + 1

# --- STEP 4: VERIFICATION ---
print(f"\nStep 3 & 4 Complete!")
print(f"Total Vocabulary Size: {vocab_size}")

# Let's check a few word mappings to see if it worked
print("\nSample Word IDs:")
check_words = ['دل', 'زندگی', 'محبت', 'غم']
for word in check_words:
    if word in tokenizer.word_index:
        print(f"'{word}' is ID: {tokenizer.word_index[word]}")
    else:
        print(f"'{word}' not found in verses.")

# IMPORTANT: Save this tokenizer! 
# You (and Person A) will need this exact file later.
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("\n'tokenizer.pkl' has been saved. Download this file from the 'Output' section!")

2025-11-26 05:48:10.789137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764136090.999524      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764136091.050633      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Step 2 Complete: Extracted 21068 clean verses.

Step 3 & 4 Complete!
Total Vocabulary Size: 10508

Sample Word IDs:
'دل' is ID: 14
'زندگی' is ID: 100
'محبت' is ID: 116
'غم' is ID: 44

'tokenizer.pkl' has been saved. Download this file from the 'Output' section!


**2.2 step 5 and 6, Sequences & Padding**

In [6]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- STEP 5: SEQUENCE GENERATION (N-grams) ---
input_sequences = []

for line in all_verses:
    # Convert line of text to line of numbers
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    # Create n-gram sequences
    # If line is [14, 50, 44], we create: [14, 50] and [14, 50, 44]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Created {len(input_sequences)} training sequences.")

# --- STEP 6: PADDING ---
# We need all sequences to be the same length for the neural network.
# We find the longest verse in your dataset first.
max_sequence_len = max([len(x) for x in input_sequences])

# Pad sequences with 0s at the beginning ('pre') so they are all length 'max_sequence_len'
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

print(f"Padded all sequences to length {max_sequence_len}.")
print(f"Shape of data tensor: {input_sequences.shape}")


Created 152146 training sequences.
Padded all sequences to length 25.
Shape of data tensor: (152146, 25)


**2.2 --- Step 7 train validation test split**

In [7]:
from sklearn.model_selection import train_test_split
import tensorflow.keras.utils as ku

# --- 1. SEPARATE X (Input) AND y (Label) ---
# X is everything EXCEPT the last number
X = input_sequences[:, :-1]
# y is ONLY the last number (the word we want to predict)
y = input_sequences[:, -1]

print(f"Input Shape (X): {X.shape}")
print(f"Target Shape (y): {y.shape}")

# --- 2. SPLIT THE DATA (80% - 10% - 10%) ---
# First, split into Train (80%) and Temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Now split the Temp (20%) in half -> Validation (10%) and Test (10%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# --- CHECKPOINT REPORT ANSWERS ---
print("\n--- DATA SPLIT REPORT ---")
print(f"Training Set:   {X_train.shape[0]} sequences (80%)")
print(f"Validation Set: {X_val.shape[0]} sequences (10%)")
print(f"Testing Set:    {X_test.shape[0]} sequences (10%)")

# Double check the percentage
total = len(input_sequences)

Input Shape (X): (152146, 24)
Target Shape (y): (152146,)

--- DATA SPLIT REPORT ---
Training Set:   121716 sequences (80%)
Validation Set: 15215 sequences (10%)
Testing Set:    15215 sequences (10%)


**fixing sequence length here**

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# --- BASELINE SEQUENCE LENGTH TO 20 ---
MAX_SEQ_LEN = 20 

# Re-pad the original sequences to length 20
input_sequences_baseline = np.array(pad_sequences(input_sequences, maxlen=MAX_SEQ_LEN, padding='pre'))

# Re-create X and y
X_base = input_sequences_baseline[:, :-1]
y_base = input_sequences_baseline[:, -1]

# Re-split into Train/Val/Test (80-10-10)
X_train_b, X_temp_b, y_train_b, y_temp_b = train_test_split(X_base, y_base, test_size=0.2, random_state=42)
X_val_b, X_test_b, y_val_b, y_test_b = train_test_split(X_temp_b, y_temp_b, test_size=0.5, random_state=42)

print(f"Baseline Data Ready: Sequence Length fixed to {MAX_SEQ_LEN}.")
print(f"X_train shape: {X_train_b.shape}")

Baseline Data Ready: Sequence Length fixed to 20.
X_train shape: (121716, 19)


**Phase 3: RNN + Adam**

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# --- BASELINE CONFIGURATION ---
VOCAB_SIZE = 10508       
MAX_SEQ_LEN = 20         
EMBEDDING_DIM = 100
RNN_UNITS = 150
DROPOUT_RATE = 0.2       
LEARNING_RATE = 0.001   
EPOCHS = 30              

# --- 1. BUILD THE 2-LAYER BASELINE MODEL ---
model_baseline = Sequential()

# Layer 1: Embedding
model_baseline.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQ_LEN-1))

# Layer 2: First SimpleRNN Layer
# return_sequences=True is REQUIRED to stack another RNN layer on top
model_baseline.add(SimpleRNN(units=RNN_UNITS, return_sequences=True))
model_baseline.add(Dropout(DROPOUT_RATE)) 

# Layer 3: Second SimpleRNN Layer
# return_sequences=False because this is the last RNN layer before output
model_baseline.add(SimpleRNN(units=RNN_UNITS, return_sequences=False))
model_baseline.add(Dropout(DROPOUT_RATE))

# Layer 4: Output
model_baseline.add(Dense(VOCAB_SIZE, activation='softmax'))

# --- 2. COMPILE ---
optimizer = Adam(learning_rate=LEARNING_RATE)
model_baseline.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model_baseline.summary()

# --- 3. TRAIN WITH EARLY STOPPING ---
# Patience=5: Stops if validation loss doesn't improve for 5 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\nStarting Baseline Training (RNN + Adam)...")
# Note: Ensure you use the '_b' variables from Step 1 (X_train_b, etc.)
history_baseline = model_baseline.fit(
    X_train_b, y_train_b, 
    epochs=EPOCHS, 
    batch_size=128,           # Baseline Requirement
    validation_data=(X_val_b, y_val_b),
    callbacks=[early_stop]
)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1764136140.410860      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1764136140.411527      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...


Starting Baseline Training (RNN + Adam)...
Epoch 1/30


I0000 00:00:1764136143.092962     124 service.cc:148] XLA service 0x7e8d8801bed0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764136143.093506     124 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1764136143.093522     124 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1764136143.556075     124 cuda_dnn.cc:529] Loaded cuDNN version 90300


 16/951 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.0121 - loss: 9.0043  

I0000 00:00:1764136146.385377     124 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


951/951 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.0402 - loss: 7.0935 - val_accuracy: 0.0473 - val_loss: 6.6839
Epoch 2/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0471 - loss: 6.5836 - val_accuracy: 0.0649 - val_loss: 6.5454
Epoch 3/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0732 - loss: 6.2679 - val_accuracy: 0.0866 - val_loss: 6.4483
Epoch 4/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0933 - loss: 5.9740 - val_accuracy: 0.0901 - val_loss: 6.4265
Epoch 5/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.1059 - loss: 5.7228 - val_accuracy: 0.0921 - val_loss: 6.4493
Epoch 6/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.1210 - loss: 5.4877 - val_accuracy: 0.0950 - val_loss: 6.4978
Epoch 7/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.1353 - loss: 5.2755 - val_accuracy: 0.0951 - val_loss: 6.5509
Epoch 8/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.1498 - loss: 5.0817 - val_accuracy: 0.0967 - va

In [10]:
from tensorflow.keras.models import save_model

# --- 1. SAVE THE ADAM MODEL (EXP 1) ---
model_baseline.save("model_rnn_adam.keras")
print("Saved 'model_rnn_adam.keras'")

Saved 'model_rnn_adam.keras'


**Phase 3: RNN + RMSprop**

In [ ]:
from tensorflow.keras.models import save_model

# --- HELPER FUNCTION TO BUILD FRESH MODEL ---
def build_baseline_rnn():
    model = Sequential()
    model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQ_LEN-1))
    model.add(SimpleRNN(units=RNN_UNITS, return_sequences=True))
    model.add(Dropout(DROPOUT_RATE))
    model.add(SimpleRNN(units=RNN_UNITS, return_sequences=False))
    model.add(Dropout(DROPOUT_RATE))
    model.add(Dense(VOCAB_SIZE, activation='softmax'))
    return model

# ==========================================
# EXPERIMENT 2: RNN + RMSprop
# ==========================================
print("\n--- Starting Experiment 2: RNN + RMSprop ---")
model_rmsprop = build_baseline_rnn()

# Compile with RMSprop
from tensorflow.keras.optimizers import RMSprop
optimizer_rms = RMSprop(learning_rate=0.001) # Baseline LR
model_rmsprop.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer_rms, metrics=['accuracy'])

# Train
history_rmsprop = model_rmsprop.fit(
    X_train_b, y_train_b, 
    epochs=30, 
    batch_size=128, 
    validation_data=(X_val_b, y_val_b),
    callbacks=[early_stop], # Same patience=5
    verbose=1
)
model_rmsprop.save("model_rnn_rmsprop.keras")


--- Starting Experiment 2: RNN + RMSprop ---
Epoch 1/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 26s 20ms/step - accuracy: 0.0411 - loss: 7.1472 - val_accuracy: 0.0452 - val_loss: 6.9029
Epoch 2/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.0425 - loss: 6.9079 - val_accuracy: 0.0517 - val_loss: 6.7155
Epoch 3/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.0520 - loss: 6.6955 - val_accuracy: 0.0674 - val_loss: 6.5797
Epoch 4/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.0672 - loss: 6.5553 - val_accuracy: 0.0760 - val_loss: 6.4927
Epoch 5/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.0781 - loss: 6.4372 - val_accuracy: 0.0849 - val_loss: 6.4405
Epoch 6/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.0888 - loss: 6.3420 - val_accuracy: 0.0873 - val_loss: 6.4033
Epoch 7/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.0956 - loss: 6.2654 - val_accuracy: 0.0919 - val_loss: 6.3752
Epoch 8/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 

**Phase 3: RNN + SGD**

In [ ]:
from tensorflow.keras.optimizers import SGD

# ==========================================
# EXPERIMENT 3:  SGD
# ==========================================
print("\n--- Starting Experiment 3 SGD ---")

# 1. Build a fresh model (Blank Brain)
model_sgd = build_baseline_rnn()

# 2. Compile with SGD
optimizer_sgd = SGD(learning_rate=0.01, momentum=0.9) 

model_sgd.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer_sgd, metrics=['accuracy'])

# 3. Train
history_sgd = model_sgd.fit(
    X_train_b, y_train_b, 
    epochs=30, 
    batch_size=128, 
    validation_data=(X_val_b, y_val_b),
    callbacks=[early_stop],
    verbose=1
)

# 4. Save
model_sgd.save("model_rnn_sgd.keras")
print("Experiment 3 Complete. Model Saved.")


--- Starting Experiment 3 SGD ---
Epoch 1/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.0396 - loss: 7.4894 - val_accuracy: 0.0452 - val_loss: 6.8105
Epoch 2/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0414 - loss: 6.8141 - val_accuracy: 0.0452 - val_loss: 6.7927
Epoch 3/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0402 - loss: 6.7889 - val_accuracy: 0.0452 - val_loss: 6.7788
Epoch 4/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0411 - loss: 6.7787 - val_accuracy: 0.0452 - val_loss: 6.7824
Epoch 5/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0413 - loss: 6.7697 - val_accuracy: 0.0452 - val_loss: 6.7812
Epoch 6/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0408 - loss: 6.7633 - val_accuracy: 0.0452 - val_loss: 6.7818
Epoch 7/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0426 - loss: 6.7629 - val_accuracy: 0.0452 - val_loss: 6.7836
Epoch 8/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0

**Phase 3: LSTM with Adam, SGD, RMS**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop, SGD

# --- BASELINE SETTINGS ---
MAX_SEQ_LEN = 20         
VOCAB_SIZE = 10508       
EMBEDDING_DIM = 100
RNN_UNITS = 150
DROPOUT_RATE = 0.2       

# --- HELPER FUNCTION: BUILD BASELINE LSTM ---
def build_baseline_lstm():
    model = Sequential()
    # Embedding: input_length is 19 because we use 19 words to predict the 20th
    model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQ_LEN-1))
    
    # Layer 1: LSTM (Stacked)
    model.add(LSTM(units=RNN_UNITS, return_sequences=True))
    model.add(Dropout(DROPOUT_RATE)) 
    
    # Layer 2: LSTM (Output)
    model.add(LSTM(units=RNN_UNITS, return_sequences=False))
    model.add(Dropout(DROPOUT_RATE)) 
    
    # Output Layer
    model.add(Dense(VOCAB_SIZE, activation='softmax'))
    return model

# ==========================================
# EXPERIMENT 4: LSTM + ADAM
# ==========================================
print("\n--- Starting Experiment 4: LSTM + Adam ---")
model_lstm_adam = build_baseline_lstm()
optimizer_adam = Adam(learning_rate=0.001) 
model_lstm_adam.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer_adam, metrics=['accuracy'])

history_lstm_adam = model_lstm_adam.fit(
    X_train_b, y_train_b, 
    epochs=30, 
    batch_size=128, 
    validation_data=(X_val_b, y_val_b),
    callbacks=[early_stop],
    verbose=1
)
model_lstm_adam.save("model_lstm_adam.keras")


# ==========================================
# EXPERIMENT 5: LSTM + RMSprop
# ==========================================
print("\n--- Starting Experiment 5: LSTM + RMSprop ---")
model_lstm_rms = build_baseline_lstm()
optimizer_rms = RMSprop(learning_rate=0.001) 
model_lstm_rms.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer_rms, metrics=['accuracy'])

history_lstm_rms = model_lstm_rms.fit(
    X_train_b, y_train_b, 
    epochs=30, 
    batch_size=128, 
    validation_data=(X_val_b, y_val_b),
    callbacks=[early_stop],
    verbose=1
)
model_lstm_rms.save("model_lstm_rmsprop.keras")


# ==========================================
# EXPERIMENT 6: LSTM + SGD
# ==========================================
print("\n--- Starting Experiment 6: LSTM + SGD ---")
model_lstm_sgd = build_baseline_lstm()
optimizer_sgd = SGD(learning_rate=0.01, momentum=0.9) 
model_lstm_sgd.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer_sgd, metrics=['accuracy'])

history_lstm_sgd = model_lstm_sgd.fit(
    X_train_b, y_train_b, 
    epochs=30, 
    batch_size=128, 
    validation_data=(X_val_b, y_val_b),
    callbacks=[early_stop],
    verbose=1
)
model_lstm_sgd.save("model_lstm_sgd.keras")

print("\n--- ALL LSTM EXPERIMENTS COMPLETE ---")


--- Starting Experiment 4: LSTM + Adam ---
Epoch 1/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.0424 - loss: 7.1282 - val_accuracy: 0.0455 - val_loss: 6.6727
Epoch 2/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.0453 - loss: 6.5800 - val_accuracy: 0.0518 - val_loss: 6.6331
Epoch 3/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.0499 - loss: 6.4586 - val_accuracy: 0.0643 - val_loss: 6.5471
Epoch 4/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.0664 - loss: 6.2977 - val_accuracy: 0.0722 - val_loss: 6.5064
Epoch 5/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.0741 - loss: 6.1664 - val_accuracy: 0.0776 - val_loss: 6.4779
Epoch 6/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.0829 - loss: 6.0122 - val_accuracy: 0.0824 - val_loss: 6.4661
Epoch 7/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.0918 - loss: 5.8822 - val_accuracy: 0.0866 - val_loss: 6.4712
Epoch 8/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 10s 11

**Transformer fuctions**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# --- 1. POSITIONAL EMBEDDING LAYER (Unchanged) ---
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

# --- 2. TRANSFORMER BLOCK LAYER (FIXED) ---
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        # 1. Attention logic
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # 2. Feed Forward logic
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

**Transformer + Adam SGD RMSprop**

In [ ]:
# --- CONFIGURATION  ---
VOCAB_SIZE = 10508
MAX_SEQ_LEN = 20
EMBEDDING_DIM = 100 
NUM_HEADS = 4       # Baseline
FF_DIM = 512        # Baseline
NUM_BLOCKS = 2      # Baseline
DROPOUT_RATE = 0.2

# --- HELPER: BUILD TRANSFORMER MODEL ---
def build_transformer_model():
    inputs = layers.Input(shape=(MAX_SEQ_LEN-1,))
    
    # 1. Add Positional Embedding
    embedding_layer = TokenAndPositionEmbedding(MAX_SEQ_LEN-1, VOCAB_SIZE, EMBEDDING_DIM)
    x = embedding_layer(inputs)
    
    # 2. Add Transformer Blocks (Stacked)
    # We add 2 blocks as per baseline
    transformer_block1 = TransformerBlock(EMBEDDING_DIM, NUM_HEADS, FF_DIM, rate=DROPOUT_RATE)
    x = transformer_block1(x)
    transformer_block2 = TransformerBlock(EMBEDDING_DIM, NUM_HEADS, FF_DIM, rate=DROPOUT_RATE)
    x = transformer_block2(x)
    
    # 3. Output Part
    x = layers.GlobalAveragePooling1D()(x) # Flatten the output
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(20, activation="relu")(x) # Small dense layer
    outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

# ==========================================
# EXPERIMENT 7: TRANSFORMER + ADAM
# ==========================================
print("\n--- Experiment 7: Transformer + Adam ---")
model_trans_adam = build_transformer_model()
model_trans_adam.compile(loss="sparse_categorical_crossentropy", optimizer=Adam(learning_rate=0.001), metrics=["accuracy"])
# Note: Transformers are complex, they might need more epochs, but we stick to baseline for comparison
history_trans_adam = model_trans_adam.fit(X_train_b, y_train_b, batch_size=128, epochs=30, validation_data=(X_val_b, y_val_b), callbacks=[early_stop])
model_trans_adam.save("model_transformer_adam.keras")

# ==========================================
# EXPERIMENT 8: TRANSFORMER + RMSprop
# ==========================================
print("\n--- Experiment 8: Transformer + RMSprop ---")
model_trans_rms = build_transformer_model()
model_trans_rms.compile(loss="sparse_categorical_crossentropy", optimizer=RMSprop(learning_rate=0.001), metrics=["accuracy"])
history_trans_rms = model_trans_rms.fit(X_train_b, y_train_b, batch_size=128, epochs=30, validation_data=(X_val_b, y_val_b), callbacks=[early_stop])
model_trans_rms.save("model_transformer_rmsprop.keras")

# ==========================================
# EXPERIMENT 9: TRANSFORMER + SGD
# ==========================================
print("\n--- Experiment 9: Transformer + SGD ---")
model_trans_sgd = build_transformer_model()
model_trans_sgd.compile(loss="sparse_categorical_crossentropy", optimizer=SGD(learning_rate=0.01, momentum=0.9), metrics=["accuracy"])
history_trans_sgd = model_trans_sgd.fit(X_train_b, y_train_b, batch_size=128, epochs=30, validation_data=(X_val_b, y_val_b), callbacks=[early_stop])
model_trans_sgd.save("model_transformer_sgd.keras")

print("\n--- ALL 9 EXPERIMENTS COMPLETED! ---")


--- Experiment 7: Transformer + Adam ---
Epoch 1/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - accuracy: 0.0389 - loss: 7.1696 - val_accuracy: 0.0452 - val_loss: 6.8238
Epoch 2/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.0430 - loss: 6.7685 - val_accuracy: 0.0452 - val_loss: 6.8900
Epoch 3/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.0420 - loss: 6.7629 - val_accuracy: 0.0452 - val_loss: 6.9045
Epoch 4/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.0434 - loss: 6.7474 - val_accuracy: 0.0452 - val_loss: 6.9266
Epoch 5/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.0440 - loss: 6.7470 - val_accuracy: 0.0452 - val_loss: 6.9345
Epoch 6/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.0435 - loss: 6.7426 - val_accuracy: 0.0452 - val_loss: 6.9515

--- Experiment 8: Transformer + RMSprop ---
Epoch 1/30
951/951 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.0410 - loss: 7.0785 - val_accuracy: 0.0452 - val_loss: 6.7966
Epo

**loading tokenizers and files for rerunning session**

In [1]:
import os
import pickle

# --- STEP 1: LOAD TOKENIZER ---
print("Searching for tokenizer.pkl...")

tokenizer_path = None

# This loop looks inside the "project" folder you just created
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('tokenizer.pkl'):
            tokenizer_path = os.path.join(root, file)
            print(f"Found it at: {tokenizer_path}")
            break

if tokenizer_path:
    with open(tokenizer_path, 'rb') as f:
        tokenizer = pickle.load(f)
    
    # Calculate vocab size again to be sure
    vocab_size = len(tokenizer.word_index) + 1
    print(f"✅ Success! Tokenizer loaded.")
    print(f"Vocabulary Size: {vocab_size}")
else:
    print("❌ ERROR: Could not find 'tokenizer.pkl'.")
    print("Please check if you uploaded the file correctly in your 'project' dataset.")

Searching for tokenizer.pkl...
Found it at: /kaggle/input/project/tokenizer.pkl


2025-11-29 09:16:35.916759: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764407796.145801      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764407796.208876      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✅ Success! Tokenizer loaded.
Vocabulary Size: 10508


**text generation for RNN and LSTM**

In [4]:
import pandas as pd
import os
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

# --- CONFIGURATION ---
MAX_SEQ_LEN = 20
SEEDS = ["محبت", "دل", "شام", "یاد", "خوشی"]
TEMPS = [0.7, 1.0, 1.3]

# --- GENERATOR FUNCTION ---
def generate_poetry(model, seed_text, next_words=12, temperature=1.0):
    generated_text = seed_text
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([generated_text])[0]
        token_list = pad_sequences([token_list], maxlen=MAX_SEQ_LEN-1, padding='pre')
        predictions = model.predict(token_list, verbose=0)[0]
        predictions = np.log(predictions + 1e-7) / temperature
        exp_preds = np.exp(predictions)
        predictions = exp_preds / np.sum(exp_preds)
        predicted_id = np.random.choice(len(predictions), p=predictions)
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_id:
                output_word = word
                break
        generated_text += " " + output_word
    return generated_text

# --- RUNNING FOR RNN & LSTM ONLY ---
results_part1 = []
print("🚀 STARTING BATCH 1 (RNN & LSTM)...")

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        # Filter for RNN/LSTM files only
        if file.endswith('.keras') and "transformer" not in file: 
            print(f"\nProcessing: {file}...")
            full_path = os.path.join(root, file)
            
            try:
                # Standard Load
                model = load_model(full_path)
                
                for seed in SEEDS:
                    for temp in TEMPS:
                        text = generate_poetry(model, seed, next_words=12, temperature=temp)
                        results_part1.append({
                            "Model": file,
                            "Seed": seed,
                            "Temperature": temp,
                            "Generated_Text": text
                        })
            except Exception as e:
                print(f"❌ Error on {file}: {e}")

# Save Part 1
df1 = pd.DataFrame(results_part1)
df1.to_csv("results_part1.csv", index=False)
print(f"\n✅ PART 1 DONE! Saved {len(df1)} poems (RNN/LSTM).")

🚀 STARTING BATCH 1 (RNN & LSTM)...

Processing: model_rnn_sgd.keras...

Processing: model_rnn_rmsprop.keras...

Processing: model_lstm_sgd.keras...

Processing: model_lstm_rmsprop.keras...

Processing: model_rnn_adam.keras...

Processing: model_lstm_adam.keras...

✅ PART 1 DONE! Saved 90 poems (RNN/LSTM).


**text generations for Transformers**

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

# --- 1. DEFINE TRANSFORMER CLASSES ---
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)
    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = models.Sequential([layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim)])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# --- 2. BUILDER FUNCTION ---
# Config must match your training exactly!
VOCAB_SIZE = 10508 
MAX_SEQ_LEN = 20
EMBEDDING_DIM = 100 
NUM_HEADS = 4       
FF_DIM = 512        
DROPOUT_RATE = 0.2

def build_transformer_model():
    inputs = layers.Input(shape=(MAX_SEQ_LEN-1,))
    embedding_layer = TokenAndPositionEmbedding(MAX_SEQ_LEN-1, VOCAB_SIZE, EMBEDDING_DIM)
    x = embedding_layer(inputs)
    transformer_block1 = TransformerBlock(EMBEDDING_DIM, NUM_HEADS, FF_DIM, rate=DROPOUT_RATE)
    x = transformer_block1(x)
    transformer_block2 = TransformerBlock(EMBEDDING_DIM, NUM_HEADS, FF_DIM, rate=DROPOUT_RATE)
    x = transformer_block2(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(20, activation="relu")(x)
    outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

# --- 3. RUNNING FOR TRANSFORMERS ONLY ---
results_part2 = []
target_files = ["model_transformer_adam.keras", "model_transformer_rmsprop.keras", "model_transformer_sgd.keras"]

print("🚀 STARTING BATCH 2 (Transformers)...")

for filename in target_files:
    # Find file
    file_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f == filename:
                file_path = os.path.join(root, f)
    
    if file_path:
        print(f"\nRebuilding & Loading: {filename}...")
        try:
            # Manual Build & Load
            model = build_transformer_model()
            model.load_weights(file_path)
            
            # Generate
            for seed in SEEDS:
                for temp in TEMPS:
                    text = generate_poetry(model, seed, next_words=12, temperature=temp)
                    results_part2.append({
                        "Model": filename,
                        "Seed": seed,
                        "Temperature": temp,
                        "Generated_Text": text
                    })
        except Exception as e:
            print(f"❌ Failed {filename}: {e}")

# --- 4. MERGE AND SAVE FINAL ---
df2 = pd.DataFrame(results_part2)
print(f"\n✅ PART 2 DONE! Saved {len(df2)} poems (Transformers).")

# Combine Part 1 and Part 2
if os.path.exists("results_part1.csv"):
    df1 = pd.read_csv("results_part1.csv")
    df_final = pd.concat([df1, df2], ignore_index=True)
    df_final.to_csv("FINAL_POETRY_RESULTS.csv", index=False)
    print(f"\n🎉 ALL DONE! Combined {len(df_final)} poems into 'FINAL_POETRY_RESULTS.csv'")
else:
    print("⚠️ Warning: Could not find Part 1 csv. Saved only Part 2.")
    df2.to_csv("FINAL_POETRY_RESULTS.csv", index=False)

🚀 STARTING BATCH 2 (Transformers)...

Rebuilding & Loading: model_transformer_adam.keras...

Rebuilding & Loading: model_transformer_rmsprop.keras...

Rebuilding & Loading: model_transformer_sgd.keras...

✅ PART 2 DONE! Saved 45 poems (Transformers).

🎉 ALL DONE! Combined 135 poems into 'FINAL_POETRY_RESULTS.csv'


**converting to urdu format**

In [ ]:
# --- SAVING WITH EXCEL-FRIENDLY ENCODING ---
if 'df_final' in locals():
    df_final.to_csv("FINAL_POETRY_RESULTS.csv", index=False, encoding='utf-8-sig')
    print("✅ Saved with Excel-friendly encoding (utf-8-sig).")
    
elif 'df1' in locals():
    # If you were running the split version
    df1.to_csv("results_part1.csv", index=False, encoding='utf-8-sig')
    if 'df2' in locals():
        df2.to_csv("FINAL_POETRY_RESULTS.csv", index=False, encoding='utf-8-sig')
    print("✅ Saved part files with Excel-friendly encoding.")
    
else:
    print("⚠️ Data variables missing. You might need to re-run the generation loop.")

✅ Saved with Excel-friendly encoding (utf-8-sig).


**generated text evaluations**

In [2]:
# 1. Install Java 17 (Required for LanguageTool)
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jre-headless

# 2. Set it as the default Java
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

print("✅ Java 17 installed successfully! You can now run the metrics code.")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.4 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,153 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,222 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy

In [5]:
import pandas as pd
import numpy as np
import os

# --- 1. AUTO-FIND THE CSV FILE ---
print("Searching for your results file...")
csv_path = None

# Look inside /kaggle/input for any CSV file
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        # We look for a csv file. If you have multiple, it picks the first one found.
        if file.endswith('.csv'):
            csv_path = os.path.join(root, file)
            print(f"Found file at: {csv_path}")
            break
    if csv_path: break

# --- 2. DEFINE MATH METRIC FUNCTIONS ---
def get_vocab_diversity(text):
    """Unique words / Total words [Source: 4.2]"""
    words = str(text).split()
    if len(words) == 0: return 0
    unique_words = set(words)
    return len(unique_words) / len(words)

def get_avg_word_length(text):
    """Character count per word [Source: 4.2]"""
    words = str(text).split()
    if len(words) == 0: return 0
    char_count = sum(len(w) for w in words)
    return char_count / len(words)

def get_repetition_rate(text):
    """Repeated phrases count (Trigrams) [Source: 4.2]"""
    words = str(text).split()
    if len(words) < 3: return 0
    trigrams = [" ".join(words[i:i+3]) for i in range(len(words)-2)]
    counts = pd.Series(trigrams).value_counts()
    repeats = counts[counts > 1].sum()
    return repeats / len(trigrams)

# --- 3. LOAD AND CALCULATE ---
if csv_path:
    try:
        # Try loading with different encodings
        try:
            df = pd.read_csv(csv_path, encoding='utf-8-sig')
        except:
            df = pd.read_csv(csv_path, encoding='utf-8')

        print(f"Loaded {len(df)} poems. Calculating math metrics...")

        # Apply calculations
        df['Diversity'] = df['Generated_Text'].apply(get_vocab_diversity)
        df['Avg_Word_Len'] = df['Generated_Text'].apply(get_avg_word_length)
        df['Repetition'] = df['Generated_Text'].apply(get_repetition_rate)
        
        # Placeholder for Manual Grammar Check
        df['Grammar_Check'] = "Manual"

        # --- GROUP BY MODEL ---
        summary = df.groupby('Model')[['Diversity', 'Avg_Word_Len', 'Repetition']].mean()
        
        print("\n--- QUANTITATIVE METRICS REPORT ---")
        display(summary)
        
        # Save output
        summary.to_csv("poetry_metrics_summary.csv")
        print("\n✅ Saved metrics to 'poetry_metrics_summary.csv'. Download this for your report.")

    except Exception as e:
        print(f"❌ Error reading the CSV: {e}")
else:
    print("❌ ERROR: Could not find ANY .csv file in /kaggle/input.")
    print("Please check your 'generated-text-1' dataset contains the file.")

Searching for your results file...
Found file at: /kaggle/input/generated-text-1/FINAL_POETRY_RESULTS.csv
Loaded 135 poems. Calculating math metrics...

--- QUANTITATIVE METRICS REPORT ---


,Diversity,Avg_Word_Len,Repetition
Model,,,
model_lstm_adam.keras,0.958974,3.082051,0.0
model_lstm_rmsprop.keras,0.938462,3.256410,0.0
model_lstm_sgd.keras,0.928205,3.210256,0.0
model_rnn_adam.keras,0.979487,3.148718,0.0
model_rnn_rmsprop.keras,0.953846,3.148718,0.0
model_rnn_sgd.keras,0.928205,3.138462,0.0
model_transformer_adam.keras,0.943590,3.194872,0.0
model_transformer_rmsprop.keras,0.953846,3.117949,0.0
model_transformer_sgd.keras,0.943590,3.000000,0.0



✅ Saved metrics to 'poetry_metrics_summary.csv'. Download this for your report.


**perplexity of base models**

In [7]:
import pandas as pd
import numpy as np
import re
import os
import pickle
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("🔄 INITIALIZING & RESTORING DATA...")

# --- 1. FIND & LOAD TOKENIZER (Fixes NameError) ---
tokenizer_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('tokenizer.pkl'):
            tokenizer_path = os.path.join(root, file)
            break

if tokenizer_path:
    with open(tokenizer_path, 'rb') as f:
        tokenizer = pickle.load(f)
    print(f"✅ Tokenizer loaded successfully! (Vocab: {len(tokenizer.word_index) + 1})")
else:
    raise FileNotFoundError("Could not find 'tokenizer.pkl'. Please upload it.")

# --- 2. LOAD & CLEAN POETRY DATA ---
dataset = load_dataset("ReySajju742/Urdu-Poetry-Dataset")
df = dataset['train'].to_pandas()

all_verses = []
for poem in df['content']:
    if poem:
        lines = poem.split('\n')
        for line in lines:
            # Simple cleaning
            text = re.sub(r'[a-zA-Z0-9]', '', line)
            text = re.sub(r'[()\[\]{}|\\/<>@#$%^&*_+=]', '', text)
            text = re.sub(r'\s+', ' ', text).strip()
            if len(text) > 5:
                all_verses.append(text)

# --- 3. CREATE SEQUENCES ---
input_sequences = []
for line in all_verses:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# --- 4. PAD & SPLIT ---
MAX_SEQ_LEN = 20 # Must match training config
input_sequences = np.array(pad_sequences(input_sequences, maxlen=MAX_SEQ_LEN, padding='pre'))

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# 80-10-10 Split (random_state=42 ensures consistency)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"✅ Data Restored. Validation Set Size: {len(X_val)}")
print("You can now run the Perplexity Calculation block.")

🔄 INITIALIZING & RESTORING DATA...
✅ Tokenizer loaded successfully! (Vocab: 10508)
✅ Data Restored. Validation Set Size: 15215
You can now run the Perplexity Calculation block.


In [9]:
import os
import math
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model

# --- 1. DEFINE CLASSES (Required for Transformer) ---
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)
    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = models.Sequential([layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim)])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Config for fallback build
VOCAB_SIZE = 10508 
MAX_SEQ_LEN = 20
EMBEDDING_DIM = 100 
NUM_HEADS = 4       
FF_DIM = 512        
DROPOUT_RATE = 0.2

def build_transformer_model():
    inputs = layers.Input(shape=(MAX_SEQ_LEN-1,))
    embedding_layer = TokenAndPositionEmbedding(MAX_SEQ_LEN-1, VOCAB_SIZE, EMBEDDING_DIM)
    x = embedding_layer(inputs)
    transformer_block1 = TransformerBlock(EMBEDDING_DIM, NUM_HEADS, FF_DIM, rate=DROPOUT_RATE)
    x = transformer_block1(x)
    transformer_block2 = TransformerBlock(EMBEDDING_DIM, NUM_HEADS, FF_DIM, rate=DROPOUT_RATE)
    x = transformer_block2(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(20, activation="relu")(x)
    outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

# --- 2. CALCULATE PERPLEXITY LOOP ---
results_perp = []
print("📊 STARTING PERPLEXITY CALCULATION...\n")

# Find models
model_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.keras'):
            model_files.append(os.path.join(root, file))

for model_path in model_files:
    model_name = os.path.basename(model_path)
    
    try:
        # Smart Load Logic
        try:
            model = load_model(model_path, custom_objects={
                "TokenAndPositionEmbedding": TokenAndPositionEmbedding,
                "TransformerBlock": TransformerBlock
            })
        except:
            model = build_transformer_model()
            model.load_weights(model_path)
            model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        # Evaluate on Validation Data
        loss, acc = model.evaluate(X_val, y_val, verbose=0)
        
        # Calculate Perplexity (e^loss)
        perplexity = math.exp(loss)
        
        # --- PRINT RESULT IMMEDIATELY ---
        print(f"Model: {model_name}")
        print(f"   -> Perplexity: {perplexity:.2f} (Loss: {loss:.4f})")
        print("-" * 40)
        
        results_perp.append({
            "Model": model_name,
            "Accuracy": acc,
            "Validation Loss": loss,
            "Perplexity": perplexity
        })
        
    except Exception as e:
        print(f"❌ Error testing {model_name}: {e}")

# Save CSV as backup
df_perp = pd.DataFrame(results_perp)
df_perp.to_csv("base_model_perplexity.csv", index=False)

📊 STARTING PERPLEXITY CALCULATION...



2025-11-29 11:27:45.134317: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: model_transformer_sgd.keras
   -> Perplexity: 517.31 (Loss: 6.2486)
----------------------------------------


I0000 00:00:1764415682.494197     950 service.cc:148] XLA service 0x79e45c0ef1c0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764415682.495118     950 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1764415683.216319     950 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Model: model_rnn_sgd.keras
   -> Perplexity: 879.03 (Loss: 6.7788)
----------------------------------------
Model: model_rnn_rmsprop.keras
   -> Perplexity: 552.56 (Loss: 6.3146)
----------------------------------------
Model: model_lstm_sgd.keras
   -> Perplexity: 875.99 (Loss: 6.7754)
----------------------------------------
Model: model_lstm_rmsprop.keras
   -> Perplexity: 589.94 (Loss: 6.3800)
----------------------------------------
Model: model_transformer_adam.keras
   -> Perplexity: 919.50 (Loss: 6.8238)
----------------------------------------
Model: model_rnn_adam.keras
   -> Perplexity: 618.01 (Loss: 6.4265)
----------------------------------------
Model: model_lstm_adam.keras
   -> Perplexity: 642.96 (Loss: 6.4661)
----------------------------------------
Model: model_transformer_rmsprop.keras
   -> Perplexity: 637.37 (Loss: 6.4574)
----------------------------------------
